# Sinusoidal Position Encoding

**难度:** Easy

实现《Attention Is All You Need》中的正弦位置编码。

位置编码加到词嵌入上，让模型感知 token 的位置信息。原始 Transformer 使用固定的正弦函数。

**签名:** `sinusoidal_pe(seq_len, d_model) -> Tensor`

**参数:**
- `seq_len` — 位置数量
- `d_model` — 嵌入维度（必须为偶数）

**返回:** 形状为 `(seq_len, d_model)` 的位置编码张量

**公式:**
- `PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))`
- `PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))`

**提示:**
1. `freqs = 1 / 10000^(2i/d_model)`，i 取 `range(d_model//2)`
2. `angles = pos[:, None] * freqs[None, :]` → 形状 `(seq_len, d_model//2)`
3. `pe[:, 0::2] = sin(angles)`，`pe[:, 1::2] = cos(angles)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def sinusoidal_pe(seq_len, d_model):
    # ---- Step 1: 位置索引 ----
    # pos = [0, 1, 2, ..., seq_len-1]，shape: (seq_len, 1)
    # unsqueeze(1) 是为了后续与 freqs 广播相乘
    pos = torch.arange(seq_len).float().unsqueeze(1)

    # ---- Step 2: 维度索引 ----
    # dim = [0, 2, 4, ..., d_model-2]，共 d_model//2 个
    # 每对 (2i, 2i+1) 共享同一个频率
    dim = torch.arange(0, d_model, 2).float()

    # ---- Step 3: 计算频率 ----
    # freqs = 1 / 10000^(2i/d_model)
    # 2i/d_model ∈ [0, 1)，所以 freqs ∈ (1/10000, 1]
    # i 小 → 频率大（变化快）→ 捕捉局部位置差异
    # i 大 → 频率小（变化慢）→ 捕捉全局位置信息
    freqs = 1.0 / (10000.0 ** (dim / d_model))

    # ---- Step 4: 计算角度 ----
    # angles[pos, i] = pos * freqs[i]
    # pos: (seq_len, 1) * freqs: (d_model//2,) → 广播 → (seq_len, d_model//2)
    # 每个位置在每个频率维度上的"相位角"
    angles = pos * freqs

    # ---- Step 5: 填充 sin 和 cos ----
    # 偶数维度 (0,2,4,...) 用 sin
    # 奇数维度 (1,3,5,...) 用 cos
    # 这样同一频率的 sin/cos 配对，形成类似复数的旋转表示
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(angles)  # 0::2 步长切片：取偶数列
    pe[:, 1::2] = torch.cos(angles)  # 1::2 步长切片：取奇数列
    return pe

In [ ]:
from torch_judge import check
check("sinusoidal_pe")

## 📝 核心思路总结

1. **正弦编码用不同频率的 sin/cos 表示位置**：低维度高频（局部区分），高维度低频（全局位置）
2. **偶数维用 sin，奇数维用 cos**：同一个频率的 sin 和 cos 配对，形成旋转编码
3. **无需学习，可外推到更长序列**：相比学习式位置编码，正弦编码天然支持任意长度